# Figure: Solubilities during degassing

In [ ]:
from pathlib import Path

results_directory = Path().resolve().parent / "Model_Outputs"
SAVE_FIG = True

## Import data and styling

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from helpers.plot_styles import (
    PLOTLY_TICK_LEN,
    PLOTLY_FONT,
    PLOTLY_LEGEND_FONTSIZE,
    PLOTLY_TICK_FONTSIZE,
    SAMPLE_DISPLAY_NAMES,
    TOOL_COLORS_HEX,
    TOOL_LINE_STYLE,
)
from helpers.degassing_data import load_all_systems

# --- USER INPUTS --- #
SAMPLES = ["MORB", "Kilauea", "Fuego", "Fogo"]
TOOLS  = ["DCompress", "DCompress (IM)", "EVo", "MAGEC", "SulfurX", "VolFe", "VESIcal_Iacono"]

In [ ]:
systems = load_all_systems(SAMPLES, TOOLS, results_dir=results_directory)

## Build the figure

In [ ]:
# Row definitions: (DataFrame column, y-axis label, oxygen-buffer slope factor).
Y_ROWS = [
    ("Fe3Fet_m", "log<sub>10</sub>[Fe<sup>3+</sup>/Fe<sup>2+</sup>]", 0.25),
    ("S6St_m",   "log<sub>10</sub>[S<sup>ox</sup>/S<sup>red</sup>]", 2),
]

EXPECTED = {
    "MORB":    {"fO2_min": -11.1, "fO2_max": -10,   "Fe3Fet_m": 1.7, "S6St_m": 18},
    "Kilauea": {"fO2_min": -9,    "fO2_max": -7,    "Fe3Fet_m": 1.3, "S6St_m": 15},
    "Fuego":   {"fO2_min": -11,   "fO2_max": -8.5,  "Fe3Fet_m": 2,   "S6St_m": 19},
    "Fogo":    {"fO2_min": -9,    "fO2_max": -4.5,  "Fe3Fet_m": 1.5, "S6St_m": 15},
}

n_rows, n_cols = len(Y_ROWS), len(SAMPLES)
top_titles = [SAMPLE_DISPLAY_NAMES.get(s, s) for s in SAMPLES]
subplot_titles = top_titles + [""] * ((n_rows - 1) * n_cols)

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    shared_xaxes=True, vertical_spacing=0.03, horizontal_spacing=0.05,
    subplot_titles=subplot_titles,
)

for r, (species, y_label, factor) in enumerate(Y_ROWS, start=1):
    for c, sample in enumerate(SAMPLES, start=1):
        exp = EXPECTED[sample]
        fig.add_trace(
            go.Scatter(
                mode="lines",
                x=(exp["fO2_min"], exp["fO2_max"]),
                y=(factor * exp["fO2_min"] + exp[species],
                   factor * exp["fO2_max"] + exp[species]),
                line=dict(color="#333", width=2, dash="solid"),
                showlegend=False,
            ),
            row=r, col=c,
        )
        for tool in TOOLS:
            if tool == "VESIcal_Iacono":
                continue
            df = systems.get(sample, {}).get(tool)
            if df is None or "P_bars" not in df.columns:
                continue
            if df["P_bars"].iloc[0] == 0:
                continue
            fig.add_trace(
                go.Scatter(
                    mode="lines",
                    x=df["logfO2"],
                    y=np.log10(df[species] / (1.0 - df[species])),
                    name=tool,
                    line=dict(color=TOOL_COLORS_HEX.get(tool, "#333"), width=2,
                              dash=TOOL_LINE_STYLE.get(tool, "solid")),
                    showlegend=(r == 1 and c == 1),
                ),
                row=r, col=c,
            )
            fig.add_trace(
                go.Scatter(
                    mode="markers",
                    x=[df.loc[0, "logfO2"]],
                    y=np.log10([df.loc[0, species] / (1.0 - df.loc[0, species])]),
                    marker=dict(color=TOOL_COLORS_HEX.get(tool, "#333")),
                    showlegend=False,
                ),
                row=r, col=c,
            )
    fig.update_yaxes(title_text=y_label, row=r, col=1)

for c in range(1, n_cols + 1):
    fig.update_xaxes(title_text="log<sub>10</sub>[fO<sub>2</sub>, bars]",
                     range=[0, None], row=n_rows, col=c)

legend_style_dict = dict(
    font=dict(size=PLOTLY_LEGEND_FONTSIZE),
    x=0.165, y=0.74,
    xanchor="right", yanchor="bottom",
    bgcolor="white",
    bordercolor="black",
    borderwidth=1,
)

fig.update_layout(
    height=600, width=1000,
    plot_bgcolor="white",
    margin=dict(t=40, r=30, l=60, b=50),
    font=PLOTLY_FONT,
    legend=legend_style_dict,
)
fig.update_xaxes(
    showline=True, linewidth=1, linecolor="black", mirror=True,
    ticks="outside", ticklen=PLOTLY_TICK_LEN, tickcolor="black",
    tickfont=dict(size=PLOTLY_TICK_FONTSIZE),
)
fig.update_yaxes(
    showline=True, linewidth=1, linecolor="black", mirror=True,
    ticks="outside", ticklen=PLOTLY_TICK_LEN, tickcolor="black",
    tickfont=dict(size=PLOTLY_TICK_FONTSIZE),
    rangemode="tozero",
)

if SAVE_FIG:    
    # write to source location
    fig.write_image("figures/Fig_Fe_S_speciation.png", scale=2,
                    height=600, width=1000)

fig.show()